# exp05a — Head Contribution Analysis (Analytical Ablation)

**Question:** How much of the exp04 elevation is driven by head 28 and the L11–12 cluster?

**Method:** Single forward pass per text; per-head attention extracted (same as exp04c).
Elevation computed under different head configurations — no model modification required.

| Config | Heads |
|---|---|
| all_heads | 0–31 (baseline, should match exp04) |
| no_head28 | 0–31 except 28 |
| head28_only | 28 |
| cluster2_only | 8, 27, 29, 30 |
| both_clusters | 8, 27, 28, 29, 30 |
| background | remaining 27 heads |

**Falsifier:** If removing head 28 has minimal effect on elevation, the signal is more
distributed than exp04c suggests.

**Data:** `exp01/valid_pairs.json` (high-fidelity subset, n=77, same as exp04)

In [ ]:
import os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import userdata
    from huggingface_hub import login
    login(token=userdata.get('HF_TOKEN'))

    if not os.path.exists('stego_CoT'):
        os.system('git clone -q https://github.com/annareshetnyak799-netizen/stego_CoT.git')
    os.chdir('stego_CoT')
    os.system('pip install -q transformers accelerate scipy')

MODEL_ID   = 'meta-llama/Llama-3.1-8B-Instruct'
INPUT_DIR  = 'results/exp01'
OUTPUT_DIR = 'results/exp05a'
N_MAX      = 80

TOP_HEADS  = [8, 27, 28, 29, 30]   # top-5 from exp04c

os.makedirs(OUTPUT_DIR, exist_ok=True)
print('cwd:', os.getcwd())

In [ ]:
import json
import numpy as np
import torch
import matplotlib.pyplot as plt
from scipy import stats
from transformers import AutoModelForCausalLM, AutoTokenizer

with open(f'{INPUT_DIR}/valid_pairs.json') as f:
    all_pairs = json.load(f)

hi_fid = [p for p in all_pairs if p['fidelity'] == 1.0]
print(f'Total: {len(all_pairs)}, high-fidelity: {len(hi_fid)}')

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map='auto',
    attn_implementation='eager',
)
model.eval()
model.config.use_cache = False

n_layers = model.config.num_hidden_layers
n_heads  = model.config.num_attention_heads
print(f'Layers: {n_layers}, heads: {n_heads}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
def find_sentence_starts(token_ids, plen):
    cot_ids   = list(token_ids[plen:])
    full_text = tokenizer.decode(cot_ids, skip_special_tokens=True)

    char_starts = []
    i = 0
    while i < len(full_text) and full_text[i] == '\n':
        i += 1
    if i < len(full_text):
        char_starts.append(i)
    while i < len(full_text):
        if full_text[i] == '\n':
            j = i + 1
            while j < len(full_text) and full_text[j] == '\n':
                j += 1
            if j - i >= 2 and j < len(full_text):
                char_starts.append(j)
            i = max(i + 1, j)
        else:
            i += 1

    positions = []
    found     = 0
    prev_len  = 0
    for tok_idx in range(len(cot_ids)):
        curr_len = len(tokenizer.decode(cot_ids[:tok_idx + 1], skip_special_tokens=True))
        while found < len(char_starts) and prev_len <= char_starts[found] < curr_len:
            positions.append(plen + tok_idx)
            found += 1
        prev_len = curr_len
        if found >= len(char_starts):
            break
    return positions


@torch.no_grad()
def extract_attentions(token_ids):
    ids_t   = torch.tensor([token_ids], dtype=torch.long).to(model.device)
    outputs = model(ids_t, output_attentions=True)
    attn    = [a[0].cpu().float() for a in outputs.attentions]
    del outputs, ids_t
    torch.cuda.empty_cache()
    return attn


def curves_per_head(attentions, sent_positions):
    """Returns (n_sent, n_layers, n_heads): mean attn from pos[K] to pos[:K] per head."""
    n_sent = len(sent_positions)
    result = np.zeros((n_sent, n_layers, n_heads))
    for k in range(1, n_sent):
        pos_k = sent_positions[k]
        prev  = sent_positions[:k]
        for l, attn in enumerate(attentions):
            result[k, l, :] = attn[:, pos_k, prev].mean(dim=-1).numpy()
    return result

In [ ]:
stego_curves = []
open_curves  = []
skipped      = 0

for pair in hi_fid[:N_MAX]:
    plen  = len(pair['payload'])
    s_pos = find_sentence_starts(pair['stego_ids'], pair['stego_plen'])[:plen]
    o_pos = find_sentence_starts(pair['open_ids'],  pair['open_plen'])[:plen]
    n_pos = min(len(s_pos), len(o_pos))

    if n_pos < 2:
        skipped += 1
        continue

    s_attn = extract_attentions(pair['stego_ids'])
    stego_curves.append(curves_per_head(s_attn, s_pos[:n_pos]))
    del s_attn

    o_attn = extract_attentions(pair['open_ids'])
    open_curves.append(curves_per_head(o_attn, o_pos[:n_pos]))
    del o_attn

    n = len(stego_curves)
    if n % 10 == 0:
        print(f'Processed {n} pairs (skipped {skipped})')

print(f'\nDone: {len(stego_curves)} pairs, {skipped} skipped')

In [ ]:
n_pairs = len(stego_curves)

configs = {
    'all_heads':     list(range(n_heads)),
    'no_head28':     [h for h in range(n_heads) if h != 28],
    'head28_only':   [28],
    'cluster2_only': [8, 27, 29, 30],
    'both_clusters': [8, 27, 28, 29, 30],
    'background':    [h for h in range(n_heads) if h not in TOP_HEADS],
}


def per_pair_elevation(s_curves, o_curves, head_set):
    """Mean elevation (stego - open) per pair, averaged over K>=1 and all layers."""
    diffs = [
        float(s[1:, :, head_set].mean()) - float(o[1:, :, head_set].mean())
        for s, o in zip(s_curves, o_curves)
    ]
    return np.array(diffs)


results = {}
print(f'{"Config":20s}  {"n_heads":>7}  {"mean_elev":>10}  {"t":>7}  {"p":>8}  sig')
print('-' * 65)
for name, heads in configs.items():
    diffs = per_pair_elevation(stego_curves, open_curves, heads)
    t, p = stats.ttest_1samp(diffs, 0)
    sig = '**' if p < 0.05 and diffs.mean() > 0 else 'n.s.'
    results[name] = {
        'n_heads': len(heads),
        'mean_elev': float(diffs.mean()),
        't': float(t),
        'p': float(p),
        'diffs': diffs,
    }
    print(f'{name:20s}  {len(heads):7d}  {diffs.mean():10.6f}  {t:7.2f}  {p:8.4f}  {sig}')

In [ ]:
baseline = results['all_heads']['mean_elev']

COLOR_MAP = {
    'all_heads': 'steelblue',
    'head28_only': 'crimson',
    'cluster2_only': 'orange',
    'both_clusters': 'purple',
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: elevation by config
ax = axes[0]
names = list(results.keys())
means = [results[n]['mean_elev'] for n in names]
sems = [results[n]['diffs'].std() / np.sqrt(n_pairs) for n in names]
colors = [COLOR_MAP.get(n, 'grey') for n in names]
ax.bar(names, means, yerr=sems, color=colors, capsize=4)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_xticks(range(len(names)))
ax.set_xticklabels(names, rotation=30, ha='right', fontsize=9)
ax.set_ylabel('Mean elevation (stego − open)')
ax.set_title('Elevation by head configuration')

# Right: fractional contribution (% of baseline)
ax2 = axes[1]
frac_names = ['head28_only', 'cluster2_only', 'both_clusters', 'background']
fracs = [results[n]['mean_elev'] / baseline * 100 for n in frac_names]
fcolors = [COLOR_MAP.get(n, 'grey') for n in frac_names]
ax2.bar(frac_names, fracs, color=fcolors)
ax2.axhline(100, color='steelblue', linestyle='--', linewidth=1, label='baseline (all heads)')
ax2.axhline(0, color='black', linewidth=0.8)
ax2.set_xticks(range(len(frac_names)))
ax2.set_xticklabels(frac_names, rotation=20, ha='right', fontsize=9)
ax2.set_ylabel('% of baseline elevation')
ax2.set_title('Fractional contribution to elevation')
ax2.legend()

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/head_contribution.png', dpi=150)
plt.show()

drop_pct = (baseline - results['no_head28']['mean_elev']) / baseline * 100
print(f'\nRemoving head 28 reduces elevation by {drop_pct:.1f}%')
print(f'Head 28 alone:    {results["head28_only"]["mean_elev"] / baseline * 100:.1f}% of baseline')
print(f'Both clusters:    {results["both_clusters"]["mean_elev"] / baseline * 100:.1f}% of baseline')
print(f'Background heads: {results["background"]["mean_elev"] / baseline * 100:.1f}% of baseline')

In [ ]:
summary = {
    'model':     MODEL_ID,
    'n_pairs':   n_pairs,
    'n_skipped': skipped,
    'baseline_elev': round(baseline, 6),
    'configs': {
        name: {
            'n_heads':   r['n_heads'],
            'mean_elev': round(r['mean_elev'], 6),
            'pct_of_baseline': round(r['mean_elev'] / baseline * 100, 1),
            't':         round(r['t'], 3),
            'p':         round(r['p'], 6),
        }
        for name, r in results.items()
    },
}

out_path = f'{OUTPUT_DIR}/exp05a_summary.json'
with open(out_path, 'w') as f:
    json.dump(summary, f, indent=2)
print('Saved:', out_path)

if IN_COLAB:
    from google.colab import drive
    import shutil, pathlib
    drive.mount('/content/drive')
    dst = pathlib.Path('/content/drive/MyDrive/stego_cot_results/exp05a')
    dst.mkdir(parents=True, exist_ok=True)
    for fp in pathlib.Path(OUTPUT_DIR).iterdir():
        shutil.copy(fp, dst / fp.name)
        print(f'  -> Drive: {fp.name}')